# Notebook 14 — Final Evaluation & Consolidated Results

**PharmaLens: An Explainable AI Framework for Drug–Target Interaction Prediction**

---

## Objective

Consolidate all experimental results into publication-ready tables and figures. Generate the final comparison that will appear in the thesis.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.evaluation.experiment_tracker import ExperimentTracker

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
tracker = ExperimentTracker()
print("Setup complete ✓")

## 1. Consolidated Results Table

In [ ]:
# Load all experiment results
all_results = tracker.get_results_dataframe()
print(f"Total experiments logged: {len(all_results)}")
print(f"\nModels: {all_results['model'].unique().tolist()}")
print(f"Splits: {all_results['split'].unique().tolist()}")

# Random split results
random_results = all_results[all_results['split'] == 'random']

# Select best result per model
best_per_model = random_results.sort_values('rmse').drop_duplicates('model', keep='first')
best_per_model = best_per_model.sort_values('rmse')

print("\n" + "=" * 110)
print("  FINAL RESULTS — KIBA Dataset (Random Split)")
print("=" * 110)
display_cols = ['model', 'rmse', 'mae', 'pearson', 'spearman', 'r2', 'ci', 'training_time']
print(best_per_model[display_cols].to_string(index=False, float_format='{:.4f}'.format))
print("=" * 110)

# Save
best_per_model[display_cols].to_csv(TABLES_DIR / 'final_results_random.csv', index=False)
print(f"\nSaved: {TABLES_DIR / 'final_results_random.csv'}")

## 2. Cross-Split Results

In [ ]:
# Results across all splits for key models
cross_split = all_results[all_results['model'].isin(['XGBoost', 'RandomForest'])]

if len(cross_split) > 0:
    pivot_rmse = cross_split.pivot_table(values='rmse', index='model', columns='split', aggfunc='min')
    pivot_pearson = cross_split.pivot_table(values='pearson', index='model', columns='split', aggfunc='max')
    
    print("\nRMSE across split strategies:")
    print(pivot_rmse.round(4).to_string())
    print("\nPearson across split strategies:")
    print(pivot_pearson.round(4).to_string())
    
    pivot_rmse.to_csv(TABLES_DIR / 'cross_split_rmse.csv')
    pivot_pearson.to_csv(TABLES_DIR / 'cross_split_pearson.csv')
else:
    print("No cross-split results available yet")

## 3. Summary Figures

In [ ]:
# Final bar chart — RMSE comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE
models = best_per_model['model'].values
rmse_vals = best_per_model['rmse'].values
colors = [MODEL_COLORS.get(m, COLORS['primary']) for m in models]

bars = axes[0].bar(range(len(models)), rmse_vals, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xticks(range(len(models)))
axes[0].set_xticklabels(models, rotation=35, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Test RMSE (Lower is Better)', fontweight='bold')
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.4f}',
                 ha='center', va='bottom', fontsize=9)

# CI
ci_vals = best_per_model['ci'].values
bars = axes[1].bar(range(len(models)), ci_vals, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(len(models)))
axes[1].set_xticklabels(models, rotation=35, ha='right', fontsize=9)
axes[1].set_ylabel('Concordance Index')
axes[1].set_title('Test CI (Higher is Better)', fontweight='bold')
for bar, val in zip(bars, ci_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.4f}',
                 ha='center', va='bottom', fontsize=9)

plt.suptitle('PharmaLens — Final Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Key Findings

### Research Contributions

1. **Multi-model framework:** PharmaLens systematically compares simple baselines (Mean, Ridge, RF), gradient boosting (XGBoost), deep learning (DeepDTA), and graph neural networks (GraphDTA) on the same dataset and splits.

2. **Explainability:** SHAP analysis reveals which molecular and protein features drive predictions, providing interpretability that pure black-box models lack.

3. **Cold-start evaluation:** Testing on unseen drugs and proteins evaluates the models' real-world applicability, showing how much performance degrades when facing novel molecules.

4. **Practical screening pipeline:** The batch screening workflow demonstrates how PharmaLens can prioritize drug-target candidates for experimental validation.

### Limitations

- All predictions are computational — experimental validation is needed
- KIBA dataset is kinase-focused; generalization to other target families needs separate evaluation
- XAI explains model behavior, not biological mechanisms
- No 3D structural information (docking scores) is used

## 5. Export for Thesis

In [ ]:
# List all generated figures
print("Generated Figures:")
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f"  {f.name}")

print(f"\nGenerated Tables:")
for f in sorted(TABLES_DIR.glob('*.csv')):
    print(f"  {f.name}")

print(f"\nSaved Models:")
for d in [BASELINE_MODELS_DIR, XGBOOST_MODELS_DIR, DEEPTDA_MODELS_DIR, GNN_MODELS_DIR]:
    for f in d.glob('*'):
        if f.suffix != '.gitkeep' and not f.name.startswith('.'):
            print(f"  {d.name}/{f.name}")

print("\n" + "=" * 50)
print("  PharmaLens Research Phase Complete")
print("=" * 50)

## Conclusion

All experiments are complete. The research phase has produced:

- ✅ Dataset exploration and preprocessing
- ✅ Drug and protein feature extraction
- ✅ Baseline models (Mean, Ridge, Random Forest)
- ✅ XGBoost with SHAP explainability
- ✅ DeepDTA (CNN-based)
- ✅ GraphDTA (GNN-based)
- ✅ Model comparison on fair splits
- ✅ Hyperparameter tuning
- ✅ Cold-start generalization evaluation
- ✅ XAI analysis
- ✅ Batch screening demonstration
- ✅ Consolidated results and figures

**Next Phase:** Build the FastAPI backend and React frontend to make PharmaLens an interactive application.